In [3]:
import pandas as pd

# 1) Google Drive mounten


pd.set_option('display.max_rows', None)

file_paths = [
    '../../logs/old_data/ransomware_log.csv',
    '../../logs/old_data/monitor_logs_benig/filebench_netsfs_20260225_001528.csv',
    '../../logs/old_data/fileserver_log.csv',
    '../../logs/old_data/varmail_log.csv',
    '../../logs/old_data/webserver_log.csv'
]

def analyze_file(path: str):
    print(f"\n{'#'*60}")
    print(f"ANALYSE DATEI: {path}")
    print(f"{'#'*60}\n")

    df = pd.read_csv(path, low_memory=False)

    # Spaltennamen säubern + vereinheitlichen
    df.columns = df.columns.str.strip()

    # Pflichtspalten prüfen
    if 'timestamp' not in df.columns:
        print("Spalte 'timestamp' nicht gefunden.")
        return
    if 'op' not in df.columns:
        print("Spalte 'op' nicht gefunden.")
        return

    # Timestamp konvertieren (robust)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # op normalisieren (Case + Whitespaces)
    df['op_norm'] = df['op'].astype(str).str.strip().str.lower()

    # OPEN filtern (fix!)
    opens = df[df['op_norm'] == 'open']

    if opens.empty:
        print("Keine 'OPEN' Operationen gefunden.")
        return

    # Resampling pro Sekunde
    opens = opens.sort_values('timestamp').set_index('timestamp')

    open_rate = opens.resample('1s').size()
    # optional, falls du wirklich jede Sekunde im Bereich als Index willst:
    open_rate = open_rate.asfreq('1s', fill_value=0)

    print(f"Gesamtanzahl Opens: {len(opens)}")
    print(f"Durchschnitt:       {open_rate.mean():.2f} Opens/Sekunde")
    print(f"Maximum:            {int(open_rate.max())} Opens/Sekunde")

    print("-" * 40)
    print("Top 10 Sekunden (höchste OPEN-Rate):")
    print("-" * 40)
    #print(open_rate.sort_values(ascending=False).head(10))

    # Wenn du WIRKLICH alles ausgeben willst (kann riesig sein):
    # print(open_rate)

for path in file_paths:
    try:
        analyze_file(path)
    except FileNotFoundError:
        print(f"Datei nicht gefunden: {path}")
    except Exception as e:
        print(f"Fehler: {e}")

pd.reset_option('display.max_rows')



############################################################
ANALYSE DATEI: ../../logs/old_data/ransomware_log.csv
############################################################

Gesamtanzahl Opens: 14561
Durchschnitt:       14.83 Opens/Sekunde
Maximum:            62 Opens/Sekunde
----------------------------------------
Top 10 Sekunden (höchste OPEN-Rate):
----------------------------------------

############################################################
ANALYSE DATEI: ../../logs/old_data/monitor_logs_benig/filebench_netsfs_20260225_001528.csv
############################################################

Gesamtanzahl Opens: 43563
Durchschnitt:       71.53 Opens/Sekunde
Maximum:            798 Opens/Sekunde
----------------------------------------
Top 10 Sekunden (höchste OPEN-Rate):
----------------------------------------

############################################################
ANALYSE DATEI: ../../logs/old_data/fileserver_log.csv
##############################################